# Pairwise win/loss probabilities

Our first building block: use the saved FIFA men's top-32 rating points to calculate **496 unique matchups**. This notebook calculates probabilities; it does not simulate tournaments yet.

**Model assumptions:** fixed ratings, neutral venue, no draws. We borrow FIFA's expected-result curve as a simplified win-probability model. FIFA's expected result is not itself an official W/L prediction.

Sources: [FIFA rankings](https://inside.fifa.com/fifa-rankings/world-ranking/men) · [FIFA methodology](https://img.fifa.com/image/upload/edbm045h0udbwkqew35a.pdf).

Run from this repository (root or `notebooks/`). In Colab, make the repository's `data/` folder available in the working directory first. Only pandas and Python's standard library are used; no GPU is needed.


In [1]:
import json
from pathlib import Path
from itertools import combinations
import math
import pandas as pd

# Find the saved snapshot without downloading or changing it.
filename = "fifa-men-top32-2026-11-25.json"
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "data" / filename).exists()), None)
if ROOT is None:
    raise FileNotFoundError("Place the repository data/ folder in the working directory.")

snapshot = json.loads((ROOT / "data" / filename).read_text())
teams = pd.DataFrame(snapshot["teams"]).sort_values("rank").reset_index(drop=True)
assert len(teams) == 32 and teams["team"].is_unique
assert teams["rating_points"].map(math.isfinite).all()
print("Snapshot date:", snapshot["page_updated_date"])
print("Source:", snapshot["source_url"])
print("Source note:", snapshot["provenance_note"])
print(teams.to_string(index=False))


Snapshot date: 2026-11-25
Source: https://inside.fifa.com/fifa-rankings/world-ranking/men
Source note: Transcribed from current rendered FIFA table, Points column, ranks 1–32. The 18 November FIFA news article places Morocco fifth; this current table places Portugal fifth, Brazil sixth and Morocco seventh. Preserves the current table as observed, not the historical 18 November release. Cause of difference is unverified.
 rank           team  rating_points
    1          Spain        2004.48
    2      Argentina        1966.13
    3         France        1950.82
    4        England        1938.90
    5       Portugal        1801.28
    6         Brazil        1800.91
    7        Morocco        1791.35
    8        Belgium        1790.37
    9    Netherlands        1774.06
   10       Colombia        1735.90
   11        Germany        1730.22
   12    Switzerland        1726.15
   13         Mexico        1723.64
   14          Italy        1717.20
   15        Croatia        1714.22


## 1. Convert a rating difference into a probability

Let $d = R_A - R_B$. Give A a relative weight of 1 and B a relative weight of $10^{-d/600}$:

$$p_A = \frac{1}{1+10^{-d/600}}, \qquad p_B = 1-p_A.$$

The scale **600** means a 600-point advantage gives 10:1 odds. It is FIFA's specified scale, not part of the definition of a Bernoulli trial. Equal ratings give 50/50.

Later we can sample $X \sim \mathrm{Bernoulli}(p_A)$: 1 means A wins, 0 means B wins. Calculating $p_A$ now is deterministic; sampling an outcome later introduces randomness.


In [2]:
SCALE = 600.0

def win_probability(rating_a, rating_b, scale=SCALE):
    """Simplified draw-free probability that A beats B."""
    if scale <= 0:
        raise ValueError("scale must be positive")
    difference = rating_a - rating_b
    relative_weight_b = 10 ** (-difference / scale)
    return 1 / (1 + relative_weight_b)

a, b = teams.iloc[0], teams.iloc[1]
p_a = win_probability(a.rating_points, b.rating_points)
print(f"{a.team} ({a.rating_points:.2f}) vs {b.team} ({b.rating_points:.2f})")
print(f"Rating difference: {a.rating_points - b.rating_points:.2f}")
print(f"P({a.team} wins) = {p_a:.2%}")
print(f"P({b.team} wins) = {1 - p_a:.2%}")


Spain (2004.48) vs Argentina (1966.13)
Rating difference: 38.35
P(Spain wins) = 53.67%
P(Argentina wins) = 46.33%


## 2. Calculate each unique matchup once

There are 32 × 31 = 992 ordered pairs. A–B and B–A describe the same match, so we need 992 / 2 = **496 calculations**. `combinations(..., 2)` visits each unique pair once and excludes self-matches.

Each table row stores both outcomes. `p_a_loss` is also `P(B wins)` because this model has no draws. Probabilities are numbers between 0 and 1, not percentages stored as text.


In [3]:
rows = []
for a, b in combinations(teams.to_dict("records"), 2):
    p_a = win_probability(a["rating_points"], b["rating_points"], SCALE)
    rows.append({
        "team_a": a["team"], "team_b": b["team"],
        "rank_a": a["rank"], "rank_b": b["rank"],
        "rating_a": a["rating_points"], "rating_b": b["rating_points"],
        "rating_difference_a_minus_b": a["rating_points"] - b["rating_points"],
        "p_a_win": p_a, "p_a_loss": 1 - p_a,
        "scale": SCALE,
        "snapshot_date": snapshot["page_updated_date"],
    })
pairs = pd.DataFrame(rows)
print(f"Unique matchups: {len(pairs)}")
print(pairs.head(10).to_string(index=False, float_format=lambda x: f"{x:.6f}"))


Unique matchups: 496
team_a      team_b  rank_a  rank_b    rating_a    rating_b  rating_difference_a_minus_b  p_a_win  p_a_loss      scale snapshot_date
 Spain   Argentina       1       2 2004.480000 1966.130000                    38.350000 0.536727  0.463273 600.000000    2026-11-25
 Spain      France       1       3 2004.480000 1950.820000                    53.660000 0.551301  0.448699 600.000000    2026-11-25
 Spain     England       1       4 2004.480000 1938.900000                    65.580000 0.562588  0.437412 600.000000    2026-11-25
 Spain    Portugal       1       5 2004.480000 1801.280000                   203.200000 0.685639  0.314361 600.000000    2026-11-25
 Spain      Brazil       1       6 2004.480000 1800.910000                   203.570000 0.685945  0.314055 600.000000    2026-11-25
 Spain     Morocco       1       7 2004.480000 1791.350000                   213.130000 0.693794  0.306206 600.000000    2026-11-25
 Spain     Belgium       1       8 2004.480000 1790.370

## 3. Check and save the table

Check the pair count, valid probabilities, complementary outcomes and equal-rating case. Each team should appear in 31 matchups. Save full numerical precision; rounding above is only for display.


In [4]:
assert len(pairs) == math.comb(len(teams), 2) == 496
assert not pairs.duplicated(["team_a", "team_b"]).any()
assert (pairs.team_a != pairs.team_b).all()
assert pairs.p_a_win.between(0, 1).all()
assert pairs.p_a_loss.between(0, 1).all()
assert all(math.isclose(w + l, 1.0) for w, l in zip(pairs.p_a_win, pairs.p_a_loss))
counts = pd.concat([pairs.team_a, pairs.team_b]).value_counts()
assert (counts == 31).all()
assert win_probability(1600, 1600) == 0.5
assert math.isclose(win_probability(2200, 1600), 10 / 11)
assert math.isclose(win_probability(1600, 2200), 1 / 11)

output = ROOT / "data" / "pairwise-win-loss-2026-11-25.csv"
pairs.to_csv(output, index=False)
reloaded = pd.read_csv(output)
assert len(reloaded) == 496
assert all(math.isclose(a, b) for a, b in zip(pairs.p_a_win, reloaded.p_a_win))
print("Checks passed. Saved:", output.relative_to(ROOT))


Checks passed. Saved: data/pairwise-win-loss-2026-11-25.csv


## What this table means

These are **model-derived probabilities**, not observed win rates or calibrated forecasts. The source snapshot's documented date/order discrepancy is preserved rather than silently resolved. Ratings stay fixed across matchups.

For a reversed matchup, use the other outcome in the same row. No additional formula calculation is necessary. Tournament formats and match sampling are the next step, when we choose to start them.
